# Validation resolutions

This notebook performs post–record-linkage clinical validation of matched SRTR donor–EHR patient pairs to assess biological plausibility and data completeness of the matches. Specifically, we aim to:

1. Validate data availability

    - Ensure each matched patient–encounter has vital signs recorded within 30 days prior to donor recovery date, confirming adequate physiologic data coverage.

2. Confirm clinical context of donation

    - Identify whether matched patients were on invasive mechanical ventilation (IMV) at any point during hospitalization.

    - Determine exposure to inotropic support (dobutamine and/or milrinone).

3. Assess temporal consistency with donation pathway

    - Verify that IMV occurred prior to recovery date for donation after brain death (DBD).

    - For donation after circulatory death (DCD), identify evidence of withdrawal of life-sustaining therapy (WLST), characterized
    by transition from IMV ± vasopressors to room air, allowing for a longer gap between death and recovery timestamps.

4. Stratify validation by donor type

    - Apply pathway-specific validation logic separately for DBD vs DCD donors.

5. Support match quality assessment

    - Generate validation flags that can be used to screen, audit, and stratify matches by confidence and clinical plausibility in downstream analyses.

# 1. Setup and Imports

In [ ]:
import polars as pl
import pandas as pd
import duckdb
import numpy as np
from datetime import datetime, timedelta
import warnings


import os
import json
import logging
import sys
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import gc

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))
from utils.io import read_data

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


from utils.config import config
site_name = config['site_name']
tables_path = config['tables_path']
file_type = config['file_type']
project_root = config['project_root']
SRTR_data_path = config["SRTR_data_path"]
sys.path.insert(0, project_root)
print(f"Site Name: {site_name}")
print(f"Tables Path: {tables_path}")
print(f"File Type: {file_type}")
from pathlib import Path
PROJECT_ROOT = Path(config['project_root'])
UTILS_DIR = PROJECT_ROOT / "utils"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_FINAL_DIR = OUTPUT_DIR / "final"
OUTPUT_INTERMEDIATE_DIR = OUTPUT_DIR / "intermediate"

# Create the output directories if they do not exist
for dir_path in [OUTPUT_DIR, OUTPUT_FINAL_DIR, OUTPUT_INTERMEDIATE_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print("Libraries imported successfully")
print(f"Polars version: {pl.__version__}")
print(f"Pandas version: {pd.__version__}")

In [ ]:
# for vitals check 
MAX_PRE_DAYS = 30
POST_GRACE_DAYS = 2


# 2. Load Data

In [ ]:
# ------------------------------------------------------------------
# Load matched datasets
# ------------------------------------------------------------------
all_matches_df = pl.read_parquet(
    OUTPUT_INTERMEDIATE_DIR / "matches_df.parquet"
)

final_clif_df = pl.read_parquet(
    OUTPUT_INTERMEDIATE_DIR / "final_clif_data.parquet"
)

srtr_df = pl.read_parquet(
    OUTPUT_INTERMEDIATE_DIR / "final_srtr_data.parquet"
)

encounter_mapping_df = pl.read_parquet(
    OUTPUT_INTERMEDIATE_DIR / "encounter_mapping_matched.parquet"
)

wide_df = pl.read_parquet(
    OUTPUT_INTERMEDIATE_DIR / "wide_df.parquet"
)

# ------------------------------------------------------------------
# Quick sanity checks
# ------------------------------------------------------------------
print(f"Matches: {all_matches_df.shape}")
print(f"CLIF cohort: {final_clif_df.shape}")
print(f"SRTR donors: {srtr_df.shape}")
print(f"Encounter mapping: {encounter_mapping_df.shape}")
print(f"Wide events: {wide_df.shape}")

# 3. Validate vitals 

`-2≤(recovery_date−last_recorded_vital_dttm)≤30 days`


To account for minor discrepancies between date- and datetime-level documentation across data sources, we allowed a small post-recovery grace window (≤2 days) when assessing proximity of the last recorded vital sign to donor recovery date. This approach reduces spurious exclusions due to timestamp granularity while preserving clinically meaningful temporal alignment.

In [ ]:
# Join vitals timestamp into matches
matches_with_vitals = (
    all_matches_df
    .join(
        final_clif_df.select([
            "encounter_block",
            "last_recorded_vital_dttm"
        ]),
        on="encounter_block",
        how="left"
    )
)


In [ ]:
matches_with_vitals = matches_with_vitals.with_columns([
    pl.col("recovery_date").cast(pl.Datetime),
    pl.col("last_recorded_vital_dttm").cast(pl.Datetime),
])

# Compute days between last vital and recovery
matches_with_vitals = matches_with_vitals.with_columns(
    (
        pl.col("recovery_date") - pl.col("last_recorded_vital_dttm")
    ).dt.total_days().alias("days_from_last_vital_to_recovery")
)

matches_with_vitals = matches_with_vitals.with_columns(
    (
        pl.col("last_recorded_vital_dttm").is_not_null()
        & (pl.col("days_from_last_vital_to_recovery") >= -POST_GRACE_DAYS)
        & (pl.col("days_from_last_vital_to_recovery") <= MAX_PRE_DAYS)
    ).alias("has_recent_vitals")
)


matches_with_vitals = matches_with_vitals.with_columns(
    pl.when(pl.col("last_recorded_vital_dttm").is_null())
      .then(pl.lit("no_vitals"))
      .when(pl.col("days_from_last_vital_to_recovery") < -POST_GRACE_DAYS)
      .then(pl.lit("vital_too_late"))
      .when(pl.col("days_from_last_vital_to_recovery") > MAX_PRE_DAYS)
      .then(pl.lit("vital_too_old"))
      .otherwise(pl.lit("pass"))
      .alias("vitals_validation_status")
)

In [ ]:
validated_matches = matches_with_vitals.filter(
    pl.col("has_recent_vitals")
)

# 4. Life support 

In [ ]:
VASOPRESSOR_COLS = [
    "med_cont_dobutamine",
    "med_cont_dopamine",
    "med_cont_epinephrine",
    "med_cont_milrinone",
    "med_cont_norepinephrine",
    "med_cont_vasopressin",
]

INOTROPE_COLS = [
    "med_cont_dobutamine",
    "med_cont_milrinone",
]


In [ ]:
def build_imv_vaso_wlst_features(
    wide_df: pl.DataFrame,
    encounter_mapping_df: pl.DataFrame,
) -> pl.DataFrame:
    """
    Returns one row per hospitalization_id with:
      - imv_ever
      - per-vasopressor ever flags
      - any_vasopressor_ever
      - any_inotrope_ever
      - wlst_dttm
    """

    # -------------------------------------------------
    # 0) Restrict wide_df to matched hospitalizations
    # -------------------------------------------------
    mapping = encounter_mapping_df.select(
        ["hospitalization_id", "encounter_block", "patient_id", "DONOR_ID"]
    ).unique()

    df = (
        mapping
        .join(wide_df, on="hospitalization_id", how="left")
        .with_columns(
            pl.col("event_dttm").cast(pl.Datetime)
        )
    )

    # -------------------------------------------------
    # 1) IMV and room-air flags
    # -------------------------------------------------
    df = df.with_columns(
        pl.col("resp_device_category")
        .cast(pl.Utf8)
        .str.to_lowercase()
        .alias("resp_device_category_lc")
    )

    df = df.with_columns([
        (pl.col("resp_device_category_lc") == "imv")
            .fill_null(False)
            .alias("is_imv"),

        (pl.col("resp_device_category_lc").is_in(
            ["room air", "room_air", "ra"]
        ))
        .fill_null(False)
        .alias("is_room_air"),
    ])

    # -------------------------------------------------
    # 2) Vasopressor timepoint flags
    # missing == not administered
    # -------------------------------------------------
    for c in VASOPRESSOR_COLS:
        df = df.with_columns(
            pl.col(c)
            .is_not_null()
            .alias(f"{c}_on")
        )

    df = df.with_columns(
        pl.any_horizontal(
            [pl.col(f"{c}_on") for c in VASOPRESSOR_COLS]
        ).alias("any_vasopressor_on")
    )

    # -------------------------------------------------
    # 3) WLST candidate
    # IMV → room air AND no vasopressors
    # -------------------------------------------------
    df = (
        df.sort(["hospitalization_id", "event_dttm"])
          .with_columns(
              pl.col("is_imv")
              .shift(1)
              .over("hospitalization_id")
              .fill_null(False)
              .alias("prev_is_imv")
          )
          .with_columns(
              (
                  pl.col("prev_is_imv")
                  & pl.col("is_room_air")
                  & (~pl.col("any_vasopressor_on"))
              ).alias("wlst_candidate")
          )
    )

    # -------------------------------------------------
    # 4) Collapse to hospitalization_id
    # -------------------------------------------------
    agg_exprs = [
        pl.col("is_imv").any().alias("imv_ever"),
        pl.col("any_vasopressor_on").any().alias("any_vasopressor_ever"),
        pl.col("event_dttm")
            .filter(pl.col("wlst_candidate"))
            .max()
            .alias("wlst_dttm"),
    ]

    # per-drug ever flags
    for c in VASOPRESSOR_COLS:
        agg_exprs.append(
            pl.col(f"{c}_on").any().alias(f"{c}_ever")
        )

    # inotrope ever
    agg_exprs.append(
        pl.any_horizontal(
            [pl.col(f"{c}_on") for c in INOTROPE_COLS]
        ).any().alias("any_inotrope_ever")
    )

    features = (
        df.group_by("encounter_block")
          .agg(agg_exprs)
    )

    return features


In [ ]:
imv_vaso_features = build_imv_vaso_wlst_features(
    wide_df=wide_df,
    encounter_mapping_df=encounter_mapping_df,
)

# print(imv_vaso_features.select(
#     "hospitalization_id",
#     "imv_ever",
#     "any_vasopressor_ever",
#     "any_inotrope_ever",
#     "wlst_dttm"
# ).head())


In [ ]:
wide_df.filter(
    pl.col("resp_device_category")
      .cast(pl.Utf8)
      .str.to_lowercase()
      .str.contains("room")
).select("resp_device_category").unique()


A couple of issues-

WLST - first ffill device category and then identify if they moved away from imv. Its not necessary that they moved to room air. If the last value is imv, but the patient died, we can use that last recorded imv event dttm along with the vasopressor dttm as the wlst_dttm 

The current definition is very strict, patient moved from IMV → room air AND was not on vasopressors. But if the patient vasopressor were missing or off then we should compute wlst for that patient. or if the patient died (is_dead==1 and use the final_death_dttm from the final_clif_df). 